In [ ]:
import glob
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
 
DATA_DIR = "/data/aveera/results"
FIGURES_DIR = "data/aveera/results/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)
 
full_paths = sorted(glob.glob(os.path.join(DATA_DIR, "irof_full_*_test_*.csv")))
if not full_paths:
    raise FileNotFoundError(f"No irof_full_*_test_*.csv files found in {DATA_DIR}")
 
dfs = []
for p in full_paths:
    m = re.match(r"irof_full_(.+)_test_.*\.csv", os.path.basename(p))
    model_name = m.group(1)
    df = pd.read_csv(p)
    df['model'] = model_name  # already present in your files, but set explicitly just in case
    dfs.append(df)
    print(f"Loaded {p} -> model='{model_name}', {len(df)} rows")
 
combined_df = pd.concat(dfs, ignore_index=True)
 
summary_df = (
    combined_df.groupby(['model', 'method'])
    .agg(mean_irof_auc=('irof_auc', 'mean'), std_irof_auc=('irof_auc', 'std'), n_images=('irof_auc', 'count'))
    .reset_index()
)
summary_df['mean_irof_auc'] = summary_df['mean_irof_auc'].round(4)
summary_df['std_irof_auc'] = summary_df['std_irof_auc'].round(4)
print("\nCombined summary:")
print(summary_df)
 
summary_out = os.path.join(FIGURES_DIR, "irof_summary_combined.csv")
summary_df.to_csv(summary_out, index=False)
print(f"\nSaved combined summary to {summary_out}")
 
model_names = sorted(combined_df['model'].unique())
method_names = sorted(combined_df['method'].unique())
palette = plt.get_cmap("tab10")
method_colors = {method: palette(i) for i, method in enumerate(method_names)}
 
n_models = len(model_names)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5), sharey=True)
if n_models == 1:
    axes = [axes]
 
for ax, model_name in zip(axes, model_names):
    model_summary = summary_df[summary_df['model'] == model_name].sort_values('mean_irof_auc')
    bar_colors = [method_colors[m] for m in model_summary['method']]
    ax.bar(model_summary['method'], model_summary['mean_irof_auc'], color=bar_colors)
    ax.set_title(model_name)
    ax.grid(axis='y', alpha=0.3)
 
axes[0].set_ylabel("Mean IROF AUC (normalized, lower = more faithful)")
fig.suptitle("Faithfulness Comparison by Attribution Method Across Models", y=1.02)
fig.tight_layout()
out_path = os.path.join(FIGURES_DIR, "irof_bar_combined.png")
fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()
 
print(f"\nSaved combined bar chart to {out_path}")